In [ ]:
import numpy as np
import time

In [ ]:
def build_element_connectivity_matrix(n_elements):
  connectivity_array = []
  print("\nEnter the end nodes for each element separated by space (for e.g.: '1 2').")
  for i in range(n_elements):
    ends = input(f"Element {i+1}: ")
    end_nodes_array = [int(end) for end in ends.split(" ")]
    end_nodes_array.sort()
    connectivity_array.append(end_nodes_array)
  return connectivity_array

def build_stiffness_vector(n_elements):
  stiffness_vector = []
  print("\nEnter the stiffness coefficient for each element.")
  for i in range(n_elements):
    stiffness = float(input(f"Element {i+1}: "))
    stiffness_vector.append(stiffness)
  return stiffness_vector

def build_global_stiffness_matrix(n_nodes, stiffness_vector, connectivity_array):
  global_stiffness_matrix = np.zeros((n_nodes, n_nodes))
  for i in range(len(stiffness_vector)):
    nodes = connectivity_array[i]
    global_stiffness_matrix[nodes[0]-1, nodes[0]-1] += stiffness_vector[i]
    global_stiffness_matrix[nodes[0]-1, nodes[1]-1] += -stiffness_vector[i]
    global_stiffness_matrix[nodes[1]-1, nodes[0]-1] += -stiffness_vector[i]
    global_stiffness_matrix[nodes[1]-1, nodes[1]-1] += stiffness_vector[i]
  return global_stiffness_matrix

def build_force_vector(n_nodes):
  force_vector = []
  print("\nEnter the force on each node. Simply press enter (without content) if unknown.")
  for i in range(n_nodes):
    force = input(f"Node {i+1}: ")
    force_vector.append(float(force)) if force != '' else force_vector.append(None)
  return force_vector

def build_displacement_vector(n_nodes):
  displacement_vector = []
  print("\nEnter the displacement on each node. Simply press enter (without content) if unknown.")
  for i in range(n_nodes):
    displacement = input(f"Node {i+1}: ")
    displacement_vector.append(float(displacement)) if displacement != '' else displacement_vector.append(None)
  return displacement_vector

# Identify unknowns
def identify_unknowns(force_vector, displacement_vector):
  unknown_list = []
  for i in range(len(force_vector)):
    if force_vector[i] == None and displacement_vector[i] == None:
      raise Exception(f"Node {i+1} is an unknown.")
    if force_vector[i] == None:
      unknown_list.append(f'F_{i+1}')
    elif displacement_vector[i] == None:
      unknown_list.append(f'u_{i+1}')
  return unknown_list

def build_new_system_equations(force_vector, global_stiffness_matrix, displacement_vector, unknown_list):
  # y = A x
  new_y = np.zeros(len(force_vector))
  new_A = np.zeros((len(force_vector), len(force_vector)))

  for i in range(len(force_vector)):
    if force_vector[i] != None:
      new_y[i] = force_vector[i]
    else:
      new_A[i, i] = 1
    for j in range(len(displacement_vector)):
      if displacement_vector[j] != None:
        new_y[i] -= global_stiffness_matrix[i, j] * displacement_vector[j]
      else:
        new_A[i, j] = global_stiffness_matrix[i, j]
  return new_y, new_A


def FEM_direct_stiffness_method():
  n_elements = int(input("Enter the number of elements:"))
  n_nodes = int(input("Enter the number of nodes:"))
  connectivity_array = build_element_connectivity_matrix(n_elements)
  stiffness_vector = build_stiffness_vector(n_elements)
  global_stiffness_matrix = build_global_stiffness_matrix(n_nodes, stiffness_vector, connectivity_array)
  force_vector = build_force_vector(n_nodes)
  displacement_vector = build_displacement_vector(n_nodes)

  print(f"======== SYSTEM DETAILS ========\n")
  print(f"Number of elements: {n_elements}")
  print(f"Number of nodes: {n_nodes}")
  print(f"\nConnectivity array: {[f"Element{i+1}: {connectivity_array[i]}" for i in range(len(connectivity_array))]}")
  print(f"\nGlobal stiffness matrix:\n{global_stiffness_matrix}")

  print(f"======== SOLVING THE SYSTEM ========\n")
  print(f"\n(00.00 ms) Compiling unknown list and rearranging equations...")
  start_time = time.time()
  unknown_list = identify_unknowns(force_vector, displacement_vector)
  new_y, new_A = build_new_system_equations(force_vector, global_stiffness_matrix, displacement_vector, unknown_list)
  print(f"({(time.time() - start_time)*1000:.2f} ms) Solving system... ")
  solved_unknown = np.linalg.solve(new_A, new_y)
  print(f"({(time.time() - start_time)*1000:.2f} ms) Printing results...")
  for i in range(len(unknown_list)):
    print(f"{unknown_list[i]} = {solved_unknown[i]}")


In [ ]:
FEM_direct_stiffness_method()

Enter the number of elements:3
Enter the number of nodes:4

Enter the end nodes for each element separated by space (for e.g.: '1 2').
Element 1: 1 2
Element 2: 2 3
Element 3: 2 4

Enter the stiffness coefficient for each element.
Element 1: 1000
Element 2: 500
Element 3: 500

Enter the force on each node. Simply press enter (without content) if unknown.
Node 1: 
Node 2: -8000
Node 3: 
Node 4: 

Enter the displacement on each node. Simply press enter (without content) if unknown.
Node 1: 0
Node 2: 
Node 3: 0
Node 4: 0
======== SYSTEM DETAILS ========

Number of elements: 3
Number of nodes: 4

Connectivity array: ['Element1: [1, 2]', 'Element2: [2, 3]', 'Element3: [2, 4]']

Global stiffness matrix:[[ 1000. -1000.     0.     0.]
 [-1000.  2000.  -500.  -500.]
 [    0.  -500.   500.     0.]
 [    0.  -500.     0.   500.]]
======== SOLVING THE SYSTEM ========


(00.00 ms) Compiling unknown list and rearranging equations...
(0.03 ms) Solving system... 
(0.72 ms) Printing results...
F_1 = -4